<a href="https://colab.research.google.com/github/EmilMadsen20/2semopgave/blob/main/JP_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


# Indlæs data

# Upload  model_df.csv til Colab
df = pd.read_csv("model_df.csv")

print(df.head())
print(df.info())


#Valg af target
target = "churn_binary"


# Split X og y

X = df.drop(columns=[target])
y = df[target].map({"No": 0, "Yes": 1})

# Tjek target-fordeling
print("\nTarget distribution:")
print(y.value_counts(normalize=True))


# One-hot encoding
X = pd.get_dummies(X, drop_first=True)


# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=7,
    stratify=y
)


# Normalisering
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# Bygger MLP model
model = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),

    layers.Dense(64, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(32, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(16, activation="relu"),
    layers.Dropout(0.2),

    layers.Dense(1, activation="sigmoid")
])


# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


# Early stopping
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)


# Træner modellen
history = model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)


# Evaluering på testdata

y_prob = model.predict(X_test_scaled).ravel()
y_pred = (y_prob > 0.5).astype(int)

print("\nModel performance på testdata:")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred))

from sklearn.metrics import precision_score

result_01 = {
    "model": "Neural Network 1 churn efter kampagneudløb ",
    "accuracy": accuracy_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_prob)
}


# Plot loss
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training og validation loss")
plt.legend()
plt.show()


# Plot accuracy

plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training og validation accuracy")
plt.legend()
plt.show()


# Gem modellen
model.save("mlp_model.keras")


print("\nModellen er gemt som mlp_model.keras")



In [ ]:
results_df = pd.DataFrame([result_01])
results_df

,model,accuracy,f1,precision,roc_auc
0,Neural Network Model 1,0.672414,0.726619,0.711268,0.716299


In [ ]:
from tensorflow.keras import regularizers # bruges til L2 (undgår overfitting)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau #sænker learning rate hvis modellen går i stå og earlystopping stopper træning hvis modellen ikke forbedrer sig

# Model 6 - Neural Network med L2 regularisering
keras.backend.clear_session()
tf.random.set_seed(7)

model_06 = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),

    layers.Dense(64, activation=None, kernel_regularizer=regularizers.l2(0.01)),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),

    layers.Dense(32, activation=None, kernel_regularizer=regularizers.l2(0.01)),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),

    layers.Dense(16, activation=None, kernel_regularizer=regularizers.l2(0.005)),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.2),

    layers.Dense(1, activation="sigmoid")
])

# Compile model
optimizer = keras.optimizers.Adam(
    learning_rate=0.001,
    clipvalue=1.0
)

model_06.compile(
    optimizer=optimizer,
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

model_06.summary()

# Early stopping
early_stop_06 = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

# Learning rate scheduler
lr_scheduler_06 = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

# Træner model 6
history_06 = model_06.fit(
    X_train_scaled,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop_06, lr_scheduler_06],
    verbose=1
)

# Evaluering på testdata
y_prob_06 = model_06.predict(X_test_scaled).ravel()
y_pred_06 = (y_prob_06 > 0.5).astype(int)

print("\nModel 6 performance på testdata:")
print("Accuracy:", accuracy_score(y_test, y_pred_06))
print("F1:", f1_score(y_test, y_pred_06))
print("ROC AUC:", roc_auc_score(y_test, y_prob_06))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred_06))

print("\nClassification report:")
print(classification_report(y_test, y_pred_06))

result_06 = {
    "model": "Neural Network 6 churn efter kampagneudløb - L2 Regularization",
    "accuracy": accuracy_score(y_test, y_pred_06),
    "f1": f1_score(y_test, y_pred_06),
    "precision": precision_score(y_test, y_pred_06),
    "roc_auc": roc_auc_score(y_test, y_prob_06)
}

# Plot loss og auc

# Plot loss
plt.figure(figsize=(8, 5))
plt.plot(history_06.history["loss"], label="Training loss")
plt.plot(history_06.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Model 6 - Training og validation loss")
plt.legend()
plt.show()

# Plot AUC
plt.figure(figsize=(8, 5))
plt.plot(history_06.history["auc"], label="Training AUC")
plt.plot(history_06.history["val_auc"], label="Validation AUC")
plt.xlabel("Epoch")
plt.ylabel("AUC")
plt.title("Model 6 - Training og validation AUC")
plt.legend()
plt.show()


In [ ]:
results_df = pd.DataFrame([result_01,result_06])
results_df

,model,accuracy,f1,precision,roc_auc
0,Neural Network Model 1,0.672414,0.726619,0.711268,0.716299
1,Neural Network Model 6 - L2 Regularization,0.689655,0.731343,0.742424,0.755591
